# Playground — central orchestrator

Linear flow:
1. **Load** raw data
2. **Clean** (split sales / returns)
3. **Features** (weekly + return rate + calendar + lags + price + demand class + commercial)
4. **Embedding** (Gemini Embedding 2 — parquet cached)
5. **Clustering** (HDBSCAN on UMAP(emb) ⊕ demand ⊕ commercial)
6. **Train global models once** (DeepAR + NS-Transformer) and produce forecast tables
7. **Model selection** on rolling-origin: Naive / SARIMAX / Prophet / LightGBM / DeepAR / NS-Transformer
8. **Final 12-week forecast** + revenue translation
9. **Plots**
10. **Agent artifacts** for n8n

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import pandas as pd
import matplotlib.pyplot as plt

from src.tools import (
    load_raw_data, split_sales_returns,
    aggregate_weekly_sku, median_price_per_sku,
    eligible_skus_by_revenue, return_rate_features,
    add_calendar_features, add_lag_rolling_features, add_price_features,
    demand_classification, commercial_profile,
    embed_sku_descriptions, cluster_skus, cluster_summary,
    build_series_for_sku, wmape,
)
from src.tools import plots
from src.models import (
    default_registry, select_best_model,
    forecast_final_horizon, attach_revenue,
    cached_forecast_factory,
)
from agent import build_agent_tables, get_forecast_for_prompt

/opt/anaconda3/envs/columbia/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


## 1. Load — both Excel sheets

In [2]:
DATA_PATH = ROOT / "data" / "raw" / "online_retail_II.xlsx"
df = load_raw_data(DATA_PATH)
print(f"raw rows: {len(df):,}  |  sheets: {df['SourceSheet'].unique().tolist()}")

raw rows: 1,067,371  |  sheets: ['Year 2009-2010', 'Year 2010-2011']


## 2. Clean — sales (target) vs returns (feature only)

In [3]:
sales, returns = split_sales_returns(df)
print(f"sales: {len(sales):,}  |  returns: {len(returns):,}  |  unique SKUs: {sales['StockCode'].nunique():,}")

sales: 1,035,620  |  returns: 18,176  |  unique SKUs: 4,873


## 3. Features — weekly + return rate + calendar + lags + price + demand class + commercial

In [4]:
weekly_sku = aggregate_weekly_sku(sales)
sku_price = median_price_per_sku(sales)

feat = return_rate_features(weekly_sku, returns, windows=(4, 13))
feat = add_calendar_features(feat)
feat = add_lag_rolling_features(feat)
feat = add_price_features(feat, sales)
demand_cls = demand_classification(weekly_sku)
commercial = commercial_profile(sales)

TOP_N = 30
skus = eligible_skus_by_revenue(
    weekly_sku, top_n=TOP_N,
    min_active_weeks=70, min_recent_active=6, recent_window=24,
)
print(f"weekly rows: {len(weekly_sku):,}  |  feat cols: {feat.shape[1]}  |  SKUs to evaluate: {len(skus)}")
demand_cls['demand_class'].value_counts()

weekly rows: 197,558  |  feat cols: 34  |  SKUs to evaluate: 30


demand_class
erratic    3847
smooth     1026
Name: count, dtype: int64

## 4. Embedding — Gemini Embedding 2 (parquet cached)

In [5]:
EMB_PATH = ROOT / "data" / "processed" / "sku_desc_emb.parquet"
emb = embed_sku_descriptions(sales, cache_path=EMB_PATH, dim=768, task_type="CLUSTERING")
print(f"embedded SKUs: {len(emb):,}  |  dim: {len(emb['embedding'].iloc[0])}")

ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

## 5. Clustering — HDBSCAN on [UMAP(emb) ⊕ demand profile ⊕ commercial profile]

In [ ]:
clusters = cluster_skus(emb, demand_cls, commercial, umap_dim=32, min_cluster_size=20)
print(cluster_summary(clusters).head(10))

## 6. Train global models — DeepAR + NS-Transformer

Trained ONCE on the full panel (caveat: use the LAST fold's window for proper held-out evaluation).
Produces a forecast table per SKU which is then exposed to `select_best_model` via `cached_forecast_factory`.

DeepAR (gluonts) on CPU/MPS: ~1-3 min for ~30 SKUs / 50 epochs.
NS-Transformer: opt-in (slower, ~10-30 min CPU). Toggle `RUN_NST` below.

In [ ]:
RUN_DEEPAR = True
RUN_NST = False  # set True if you have time / a GPU

global_forecasts: dict[str, pd.DataFrame] = {}

if RUN_DEEPAR:
    from src.models.deepar import DeepARWrapper
    deepar = DeepARWrapper(backend="local", context_length=52, prediction_length=12)
    deepar.fit(weekly_sku, skus=skus, max_epochs=30)
    rows = []
    for sku in skus:
        try:
            yhat = deepar.forecast(sku, horizon=12)
            rows.extend((sku, h, float(v)) for h, v in enumerate(yhat, start=1))
        except Exception as e:
            print(f"[DeepAR] {sku} failed: {e}")
    global_forecasts["DeepAR"] = pd.DataFrame(rows, columns=["StockCode", "Horizon", "Forecast"])
    print(f"DeepAR forecast table: {global_forecasts['DeepAR'].shape}")

if RUN_NST:
    from src.models.ns_transformer import train_ns_transformer, predict_ns_transformer
    nst, meta = train_ns_transformer(weekly_sku, skus=skus,
                                     params={"SEQ_LEN": 52, "PRED_LEN": 12, "EPOCHS": 20})
    global_forecasts["NSTransformer"] = predict_ns_transformer(
        nst, weekly_sku, skus=skus, horizon=12, seq_len=52, label_len=12,
    )
    print(f"NS-Transformer forecast table: {global_forecasts['NSTransformer'].shape}")

## 7. Model selection — rolling-origin (3 folds × 12-week test) — ALL models in one pass

Local models retrain per (SKU, fold). Global models reuse their cached forecast table.
The final benchmark prints **WMAPE** per model (the metric we care about — volume-weighted).

In [ ]:
registry = default_registry()                                 # Naive, SARIMAX, Prophet (+ LightGBM if libomp)
for name, fc in global_forecasts.items():                     # plug DeepAR / NSTransformer
    registry[name] = cached_forecast_factory(fc)
print("Models in benchmark:", list(registry))

results = select_best_model(
    weekly_sku=weekly_sku, skus=skus, model_registry=registry,
    n_folds=3, test_size=12, block_size=4, min_train=70,
)
choices_df = results["choices"]
test_block = results["test_block"]
bench = results["benchmark"]

print("\n=== BENCHMARK (validation fold, ALL SKUs) ===")
print(bench.to_string(index=False))
if not test_block.empty:
    print("\nHeld-out test WMAPE per fold (chosen model):")
    for fold, g in test_block.groupby("Fold"):
        print(f"  fold {fold}: WMAPE = {wmape(g['Actual_Block'], g['Forecast_Block']):.4f}")
print("\nChosen model distribution:")
print(choices_df['Chosen_Model'].value_counts())

### Diagnostic — inspect a single SKU's actuals vs forecasts (sanity)
Use this if benchmark numbers look suspicious (all 0s, all the same, etc.).

In [ ]:
from src.tools.evaluation import rolling_block_evaluate
from src.tools.splits import split_train_val_test

sku = skus[0]
s = build_series_for_sku(weekly_sku, sku)
print(f"SKU {sku} — series len {len(s)} | last 12 weeks: {s.tail(12).values.astype(int).tolist()}")
split = split_train_val_test(s, val_size=12, test_size=12)
if split is not None:
    tr, va, _ = split
    for name, mf in registry.items():
        ev = rolling_block_evaluate(tr, va, mf, block_size=4)
        print(f"\n  [{name}] Forecast vs Actual on validation:")
        print("  ", ev[['Block','Actual','Forecast']].round(2).to_string(index=False))

## 8. Final 12-week forecast + revenue translation

In [ ]:
forecast_df = forecast_final_horizon(weekly_sku, choices_df, registry, horizon=12)
fc_with_rev = attach_revenue(forecast_df, sku_price)
fc_with_rev.head()

## 9. Plots

In [ ]:
if not test_block.empty:
    plots.plot_block_ape_boxplot(test_block); plt.show()
plots.plot_chosen_model_counts(choices_df); plt.show()

## 10. Agent artifacts — parquet for n8n FastAPI server

In [ ]:
agent_horizon, agent_summary = build_agent_tables(forecast_df, sku_price, choices_df)
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
agent_horizon.to_parquet(OUT / "agent_horizon.parquet", index=False)
agent_summary.to_parquet(OUT / "agent_summary.parquet", index=False)
print(f"wrote {OUT}/agent_horizon.parquet, agent_summary.parquet")

for prompt in [
    "What is the latest 12-week forecast for product 22197?",
    "Forecast SKU 85123A for the next 4 weeks",
]:
    print(get_forecast_for_prompt(prompt, agent_summary, agent_horizon))
    print("=" * 60)